# 🧠 Mineflayer + LLM Agent — Minecraft AI on Colab

Runs [Mineflayer](https://github.com/PrismarineJS/node-mineflayer) (Node.js Minecraft bot) + Ollama LLM on Colab.
The LLM can **see the world**, **decide actions**, and **execute them** via Mineflayer.

## Architecture
```
Discord Bot → POST /goal "Build a house" → Colab FastAPI
                                              ↓
                                    LLM (70B) sees world state
                                              ↓
                                    LLM decides: collect wood, craft, build
                                              ↓
                                    Mineflayer executes actions in MC
                                              ↓
                                    Loop: observe → think → act → repeat
```

**The LLM has these tools (via Mineflayer):**
- `moveTo(x, z)` — walk to coordinates (uses Baritone if available)
- `mineBlock(type)` — mine a specific block
- `placeBlock(x, y, z, type)` — place a block
- `craftItem(recipe)` — craft an item
- `attack(entity)` — attack a mob/player
- `chat(message)` — send a chat message
- `dropItem(item, count)` — drop items
- `equipItem(item)` — equip a tool/armor
- `openChest(x, y, z)` — open and interact with chests
- `getInventory()` — list inventory items
- `getNearbyBlocks(radius)` — scan nearby blocks
- `getNearbyEntities()` — list nearby mobs/players
- `getHealth()` — health, food, position

**Instructions:**
1. Runtime > Change runtime type > GPU (T4 for 7B, A100 for 70B)
2. Execute all cells in order
3. Set MC server + Ollama model in Cell 2

In [ ]:
# Cell 1: Install dependencies
!curl -fsSL https://ollama.com/install.sh | sh
!pip install fastapi uvicorn pyngrok requests
!npm install mineflayer mineflayer-pathfinder mineflayer-auto-eat mineflayer-pvp prismarine-item
print('✅ Dependencies installed')

In [ ]:
# Cell 2: Configuration
MC_VERSION = '1.21.1'
MC_SERVER = ''  # <-- Your server IP:port
MC_USERNAME = 'LLM_Bot'  # Bot username in Minecraft

LLM_MODEL = 'qwen2.5:14b'  # 7b, 14b (T4) or 32b, 70b (A100)
LLM_HOST = 'http://localhost:11434'  # Ollama runs locally on Colab

NGROK_AUTHTOKEN = ''  # <-- Paste your ngrok authtoken
BOT_WEBHOOK_URL = ''  # <-- Bot webhook URL
API_PORT = 7000

# Agent settings
MAX_ACTIONS_PER_GOAL = 50  # Safety limit
ACTION_DELAY_MS = 500  # Delay between actions
OBSERVATION_RADIUS = 32  # Block scan radius

print(f'Config: MC {MC_VERSION}, LLM={LLM_MODEL}, Server={MC_SERVER}')

In [ ]:
# Cell 3: Start Ollama + pull model
import subprocess, os, time

os.system('pkill ollama 2>/dev/null; sleep 2')
env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0:11434'
ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, env=env)
time.sleep(3)
print('⏳ Pulling LLM model...')
os.system(f'ollama pull {LLM_MODEL}')
print(f'✅ Ollama ready with {LLM_MODEL}')

In [ ]:
# Cell 4: Mineflayer bot script (Node.js)
mineflayer_script = r'''
const mineflayer = require('mineflayer');
const pathfinder = require('mineflayer-pathfinder');
const { Movements, goals } = pathfinder;
const autoEat = require('mineflayer-auto-eat');

const bot = mineflayer.createBot({
  host: process.env.MC_SERVER,
  port: parseInt(process.env.MC_PORT || '25565'),
  username: process.env.MC_USERNAME || 'LLM_Bot',
  version: process.env.MC_VERSION || '1.21.1',
  hideErrors: false,
});

// Pathfinder setup
const mcData = require('minecraft-data')(bot.version);
const defaultMove = new Movements(bot, mcData);
bot.pathfinder.setMovements(defaultMove);

// Auto-eat
bot.loadPlugin(autoEat);

// State tracking
let currentGoal = null;
let isExecuting = false;
let actionQueue = [];
let lastObservation = null;

bot.once('spawn', () => {
  console.log('SPAWNED');
  bot.chat('Hello! I am an LLM-controlled bot.');
});

bot.on('health', () => {
  if (bot.health <= 0) {
    console.log('DIED');
  }
});

bot.on('chat', (username, message) => {
  if (username === bot.username) return;
  console.log(`CHAT ${username}: ${message}`);
  // Forward chat to the API server for LLM processing
  fetch('http://localhost:' + (parseInt(process.env.API_PORT) + 1) + '/mc-chat', {
    method: 'POST',
    headers: { 'Content-Type': 'application/json' },
    body: JSON.stringify({ username, message }),
  }).catch(() => {});
});

// ─── Action executors ──────────────────────────────────────

async function executeAction(action) {
  const { type, params } = action;
  switch (type) {
    case 'moveTo': {
      const { x, z } = params;
      bot.pathfinder.setGoal(new goals.GoalBlock(x, bot.entity.position.y, z));
      await new Promise((resolve) => bot.once('goal_reached', resolve));
      return { success: true, message: `Arrived at ${x}, ${z}` };
    }
    case 'mineBlock': {
      const block = bot.findBlock({ matching: (b) => b.name.includes(params.type), maxDistance: 32 });
      if (!block) return { success: false, message: `No ${params.type} found nearby` };
      await bot.dig(block);
      return { success: true, message: `Mined ${block.name} at ${block.position}` };
    }
    case 'placeBlock': {
      const { x, y, z, type } = params;
      const item = bot.inventory.items().find(i => i.name === type);
      if (!item) return { success: false, message: `No ${type} in inventory` };
      await bot.equip(item, 'hand');
      const refBlock = bot.blockAt(new Vec3(x, y - 1, z));
      await bot.placeBlock(refBlock, new Vec3(0, 1, 0));
      return { success: true, message: `Placed ${type} at ${x},${y},${z}` };
    }
    case 'craftItem': {
      const { recipe } = params;
      const recipes = bot.recipesFor(mcData.itemsByName[recipe]?.id || 0);
      if (!recipes.length) return { success: false, message: `No recipe for ${recipe}` };
      await bot.craft(recipes[0], 1, null);
      return { success: true, message: `Crafted ${recipe}` };
    }
    case 'attack': {
      const entity = bot.nearestEntity(e => e.name === params.target || e.username === params.target);
      if (!entity) return { success: false, message: `No ${params.target} nearby` };
      bot.attack(entity);
      return { success: true, message: `Attacked ${params.target}` };
    }
    case 'chat': {
      bot.chat(params.message);
      return { success: true, message: `Said: ${params.message}` };
    }
    case 'dropItem': {
      const item = bot.inventory.items().find(i => i.name === params.item);
      if (!item) return { success: false, message: `No ${params.item} in inventory` };
      await bot.tossStack(item);
      return { success: true, message: `Dropped ${params.item}` };
    }
    case 'equipItem': {
      const item = bot.inventory.items().find(i => i.name === params.item);
      if (!item) return { success: false, message: `No ${params.item} in inventory` };
      await bot.equip(item, params.slot || 'hand');
      return { success: true, message: `Equipped ${params.item}` };
    }
    case 'jump': {
      bot.setControlState('jump', true);
      setTimeout(() => bot.setControlState('jump', false), 500);
      return { success: true, message: 'Jumped' };
    }
    case 'lookAt': {
      const { x, y, z } = params;
      await bot.lookAt(new Vec3(x, y, z));
      return { success: true, message: `Looking at ${x},${y},${z}` };
    }
    case 'stop': {
      bot.pathfinder.setGoal(null);
      bot.clearControlStates();
      return { success: true, message: 'Stopped' };
    }
    default:
      return { success: false, message: `Unknown action: ${type}` };
  }
}

// ─── World observation ──────────────────────────────────────

function getWorldState() {
  const pos = bot.entity.position;
  const nearbyBlocks = [];
  const radius = parseInt(process.env.OBSERVATION_RADIUS || '32');
  
  // Scan nearby blocks (simple version — find ores and interesting blocks)
  const directions = [[1,0,0],[-1,0,0],[0,1,0],[0,-1,0],[0,0,1],[0,0,-1]];
  for (let dx = -3; dx <= 3; dx++) {
    for (let dy = -2; dy <= 2; dy++) {
      for (let dz = -3; dz <= 3; dz++) {
        const block = bot.blockAt(new Vec3(pos.x + dx, pos.y + dy, pos.z + dz));
        if (block && block.name !== 'air' && block.name !== 'cave_air') {
          nearbyBlocks.push({ name: block.name, x: pos.x+dx, y: pos.y+dy, z: pos.z+dz });
        }
      }
    }
  }
  
  const nearbyEntities = Object.values(bot.entities)
    .filter(e => e.position.distanceTo(pos) < 32 && e !== bot.entity)
    .map(e => ({ name: e.name || e.username, type: e.type, distance: e.position.distanceTo(pos).toFixed(1) }));
  
  const inventory = bot.inventory.items().map(i => ({ name: i.name, count: i.count }));
  
  return {
    position: { x: pos.x.toFixed(1), y: pos.y.toFixed(1), z: pos.z.toFixed(1) },
    health: bot.health,
    food: bot.food,
    saturation: bot.foodSaturation,
    oxygen: bot.oxygenLevel,
    gameMode: bot.gameMode,
    isMoving: bot.entity.velocity.x !== 0 || bot.entity.velocity.z !== 0,
    nearbyBlocks: nearbyBlocks.slice(0, 50),
    nearbyEntities: nearbyEntities.slice(0, 20),
    inventory: inventory,
    timeOfDay: bot.time.timeOfDay,
    isRaining: bot.isRaining,
    biome: bot.blockAt(pos)?.biome || 'unknown',
  };
}

// ─── HTTP server for API ────────────────────────────────────
const http = require('http');

const server = http.createServer(async (req, res) => {
  res.setHeader('Content-Type', 'application/json');
  
  if (req.url === '/world' && req.method === 'GET') {
    res.end(JSON.stringify(getWorldState()));
    return;
  }
  
  if (req.url === '/status' && req.method === 'GET') {
    const pos = bot.entity?.position;
    res.end(JSON.stringify({
      connected: !!bot.entity,
      username: bot.username,
      position: pos ? { x: pos.x.toFixed(1), y: pos.y.toFixed(1), z: pos.z.toFixed(1) } : null,
      health: bot.health,
      food: bot.food,
    }));
    return;
  }
  
  if (req.url === '/action' && req.method === 'POST') {
    let body = '';
    req.on('data', chunk => body += chunk);
    req.on('end', async () => {
      try {
        const action = JSON.parse(body);
        const result = await executeAction(action);
        res.end(JSON.stringify(result));
      } catch (e) {
        res.end(JSON.stringify({ success: false, message: e.message }));
      }
    });
    return;
  }
  
  if (req.url === '/chat' && req.method === 'POST') {
    let body = '';
    req.on('data', chunk => body += chunk);
    req.on('end', () => {
      const { message } = JSON.parse(body);
      bot.chat(message);
      res.end(JSON.stringify({ success: true }));
    });
    return;
  }
  
  res.statusCode = 404;
  res.end(JSON.stringify({ error: 'Not found' }));
});

const MF_PORT = parseInt(process.env.MF_PORT || '7001');
server.listen(MF_PORT, () => console.log(`Mineflayer API on port ${MF_PORT}`));

bot.on('kicked', (reason) => console.log('KICKED:', reason));
bot.on('error', (err) => console.log('ERROR:', err.message));
bot.on('end', () => console.log('DISCONNECTED'));
'''

with open('/root/mineflayer_bot.js', 'w') as f:
    f.write(mineflayer_script)
print('✅ Mineflayer bot script written')

In [ ]:
# Cell 5: Start Mineflayer bot
import subprocess, os, time

os.environ['MC_SERVER'] = MC_SERVER.split(':')[0] if MC_SERVER else 'localhost'
os.environ['MC_PORT'] = MC_SERVER.split(':')[1] if ':' in MC_SERVER else '25565'
os.environ['MC_USERNAME'] = MC_USERNAME
os.environ['MC_VERSION'] = MC_VERSION
os.environ['MF_PORT'] = '7001'
os.environ['API_PORT'] = str(API_PORT)
os.environ['OBSERVATION_RADIUS'] = str(OBSERVATION_RADIUS)

print(f'⏳ Starting Mineflayer bot...')
print(f'   Server: {MC_SERVER}')
print(f'   Username: {MC_USERNAME}')

mf_proc = subprocess.Popen(['node', '/root/mineflayer_bot.js'],
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           env=os.environ)

# Wait for spawn
time.sleep(15)
output = mf_proc.stdout.read1(4096).decode() if mf_proc.stdout else ''
print(f'Mineflayer output: {output[:500]}')

if 'SPAWNED' in output:
    print('✅ Mineflayer bot spawned in Minecraft')
else:
    print('⏳ Waiting for spawn... (may take longer)')

In [ ]:
# Cell 6: FastAPI server — LLM agent loop
from fastapi import FastAPI
from pydantic import BaseModel
import requests, json, time, threading, asyncio

app = FastAPI(title='Mineflayer LLM Agent', version='1.0.0')

MF_API = 'http://localhost:7001'  # Mineflayer internal API
OLLAMA_API = f'{LLM_HOST}/v1/chat/completions'

class GoalRequest(BaseModel):
    goal: str  # e.g. "Build a small house", "Mine 10 iron ore", "Find diamonds"
    max_actions: int = 50

class ChatRequest(BaseModel):
    message: str

# ─── LLM prompt builder ──────────────────────────────────────

SYSTEM_PROMPT = '''You are an AI agent controlling a Minecraft bot via Mineflayer.
You receive the current world state and must decide the next action to take.

Available actions (respond with JSON only):
[{"type": "moveTo", "params": {"x": 100, "z": 200}}]
[{"type": "mineBlock", "params": {"type": "oak_log"}}]
[{"type": "placeBlock", "params": {"x": 100, "y": 64, "z": 200, "type": "oak_planks"}}]
[{"type": "craftItem", "params": {"recipe": "oak_planks"}}]
[{"type": "attack", "params": {"target": "zombie"}}]
[{"type": "chat", "params": {"message": "Hello!"}}]
[{"type": "dropItem", "params": {"item": "cobblestone"}}]
[{"type": "equipItem", "params": {"item": "iron_pickaxe"}}]
[{"type": "jump", "params": {}}]
[{"type": "lookAt", "params": {"x": 100, "y": 64, "z": 200}}]
[{"type": "stop", "params": {}}]

Rules:
1. Respond with a JSON array of actions to execute in order.
2. Each action must have "type" and "params".
3. Be efficient — don't waste actions.
4. If you can't complete the goal, explain why and stop.
5. If health is low, prioritize survival (eat, flee, stop).
6. For mining, first moveTo near the ore, then mineBlock.
7. For building, collect materials first, then place blocks.

Respond with ONLY the JSON array, no other text.'''

def get_world_state():
    try:
        resp = requests.get(f'{MF_API}/world', timeout=10)
        return resp.json()
    except Exception as e:
        return {'error': str(e)}

def execute_action(action):
    try:
        resp = requests.post(f'{MF_API}/action', json=action, timeout=30)
        return resp.json()
    except Exception as e:
        return {'success': False, 'message': str(e)}

def call_llm(world_state, goal, history=''):
    """Call Ollama LLM to decide next actions."""
    user_msg = f"""Goal: {goal}

Current world state:
{json.dumps(world_state, indent=2)}

Previous actions and results:
{history}

What actions should the bot take next? Respond with JSON array only."""

    payload = {
        'model': LLM_MODEL,
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_msg},
        ],
        'temperature': 0.3,
        'max_tokens': 1000,
        'stream': False,
    }
    
    try:
        resp = requests.post(OLLAMA_API, json=payload, timeout=120)
        if resp.status_code == 200:
            content = resp.json()['choices'][0]['message']['content']
            # Parse JSON actions from response
            # Try to extract JSON array from response
            import re
            json_match = re.search(r'\[.*\]', content, re.DOTALL)
            if json_match:
                return json.loads(json_match.group())
            return []
        return []
    except Exception as e:
        print(f'LLM error: {e}')
        return []

# ─── Agent loop ──────────────────────────────────────────────

agent_running = False
agent_log = []

def run_agent_loop(goal, max_actions=50):
    global agent_running, agent_log
    agent_running = True
    agent_log = []
    history = ''
    
    for step in range(max_actions):
        if not agent_running:
            agent_log.append('Agent stopped by user')
            break
        
        # 1. Observe world
        world = get_world_state()
        if 'error' in world:
            agent_log.append(f'World error: {world["error"]}')
            break
        
        # 2. Check survival
        if world.get('health', 20) <= 4:
            agent_log.append('⚠️ Low health — stopping agent')
            break
        
        # 3. Ask LLM what to do
        actions = call_llm(world, goal, history)
        if not actions:
            agent_log.append('LLM returned no actions — goal complete or stuck')
            break
        
        # 4. Execute actions
        for action in actions:
            if not agent_running:
                break
            result = execute_action(action)
            action_desc = f'{action["type"]}({action.get("params", {})}): {result.get("message", "")}'
            agent_log.append(f'[Step {step}] {action_desc}')
            history += f'\n{action_desc}'
            time.sleep(ACTION_DELAY_MS / 1000)
        
        # Keep history short to avoid context overflow
        if len(history) > 4000:
            history = history[-2000:]
    
    agent_running = False
    agent_log.append(f'Agent finished after {step + 1} steps')

@app.get('/health')
async def health():
    return {'status': 'ok', 'llm': LLM_MODEL, 'mc': MC_SERVER}

@app.get('/world')
async def world():
    return get_world_state()

@app.get('/status')
async def status():
    try:
        resp = requests.get(f'{MF_API}/status', timeout=10)
        data = resp.json()
        data['agent_running'] = agent_running
        data['llm_model'] = LLM_MODEL
        return data
    except:
        return {'connected': False, 'agent_running': agent_running}

@app.post('/goal')
async def set_goal(req: GoalRequest):
    """Set a high-level goal for the LLM agent to achieve."""
    global agent_running
    if agent_running:
        return {'error': 'Agent already running — stop it first'}
    
    # Run agent loop in background thread
    thread = threading.Thread(target=run_agent_loop, args=(req.goal, req.max_actions), daemon=True)
    thread.start()
    return {'success': True, 'goal': req.goal, 'max_actions': req.max_actions}

@app.post('/stop')
async def stop_agent():
    global agent_running
    agent_running = False
    # Also stop Mineflayer
    try:
        requests.post(f'{MF_API}/action', json={'type': 'stop', 'params': {}}, timeout=5)
    except:
        pass
    return {'success': True, 'message': 'Agent stopped'}

@app.get('/log')
async def get_log(lines: int = 50):
    return {'log': '\n'.join(agent_log[-lines:])}

@app.post('/chat')
async def send_chat(req: ChatRequest):
    try:
        requests.post(f'{MF_API}/chat', json={'message': req.message}, timeout=10)
        return {'success': True}
    except Exception as e:
        return {'error': str(e)}

@app.post('/action')
async def send_action(action: dict):
    """Send a single action directly (bypass LLM)."""
    return execute_action(action)

print('✅ LLM Agent API ready:')
print('  GET  /health    — API health')
print('  GET  /world     — World state from Mineflayer')
print('  GET  /status    — Bot + agent status')
print('  POST /goal      — Set high-level goal for LLM agent')
print('  POST /stop      — Stop agent loop')
print('  GET  /log       — Agent action log')
print('  POST /chat      — Send chat message in MC')
print('  POST /action    — Send single action (bypass LLM)')

In [ ]:
# Cell 7: Start ngrok + API server
from pyngrok import ngrok, conf
import nest_asyncio, threading, uvicorn

if NGROK_AUTHTOKEN:
    conf.get_default().auth_token = NGROK_AUTHTOKEN

ngrok.kill()
import time; time.sleep(2)

tunnel = ngrok.connect(API_PORT, 'http')
AGENT_URL = tunnel.public_url
print(f'🌐 LLM Agent URL: {AGENT_URL}')
print(f'   → Set MINEFLAYER_AGENT_URL={AGENT_URL} in bot .env')

if BOT_WEBHOOK_URL:
    try:
        resp = requests.post(BOT_WEBHOOK_URL, json={'url': AGENT_URL, 'type': 'mineflayer'}, timeout=10)
        print(f'📡 Bot notified: {resp.status_code}')
    except Exception as e:
        print(f'⚠️ Webhook failed: {e}')

nest_asyncio.apply()
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=API_PORT, log_level='info')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print(f'✅ API server running on port {API_PORT}')
print(f'   Test: {AGENT_URL}/health')

In [ ]:
# Cell 8: Keep-alive loop
import time, requests
from datetime import datetime, timedelta

REGENERATE_HOURS = 24
start_time = datetime.now()
regenerate_at = start_time + timedelta(hours=REGENERATE_HOURS)

print(f'🕐 LLM Agent started at {start_time.strftime("%H:%M:%S")}')
print(f'🔄 Will regenerate at {regenerate_at.strftime("%H:%M:%S")}')

ok_count = 0
fail_count = 0

try:
    while True:
        now = datetime.now()
        
        try:
            resp = requests.get(f'{AGENT_URL}/health', timeout=10)
            if resp.status_code == 200:
                ok_count += 1
                status = '✅'
            else:
                fail_count += 1
                status = f'⚠️ {resp.status_code}'
        except:
            fail_count += 1
            status = '❌'
        
        elapsed = now - start_time
        if int(elapsed.total_seconds()) % 300 == 0 and int(elapsed.total_seconds()) > 0:
            try:
                st = requests.get(f'{AGENT_URL}/status', timeout=10).json()
                mc_info = f"MC={'online' if st.get('connected') else 'offline'}"
                if st.get('position'):
                    p = st['position']
                    mc_info += f" pos=({p['x']},{p['y']},{p['z']})"
                if st.get('agent_running'):
                    mc_info += ' agent=running'
                mc_info += f" hp={st.get('health', '?')}"
            except:
                mc_info = 'MC=status?'
            print(f'[{now.strftime("%H:%M:%S")}] uptime={int(elapsed.total_seconds()/60)}min ok={ok_count} fail={fail_count} {status} {mc_info}')
        
        if now >= regenerate_at:
            print('🔄 Regenerating ngrok tunnel...')
            ngrok.kill()
            time.sleep(3)
            new_tunnel = ngrok.connect(API_PORT, 'http')
            AGENT_URL = new_tunnel.public_url
            print(f'   New URL: {AGENT_URL}')
            if BOT_WEBHOOK_URL:
                try:
                    requests.post(BOT_WEBHOOK_URL, json={'url': AGENT_URL, 'type': 'mineflayer'})
                    print('   📡 Bot notified')
                except Exception as e:
                    print(f'   ⚠️ Webhook failed: {e}')
            regenerate_at = now + timedelta(hours=REGENERATE_HOURS)
        
        time.sleep(60)
except KeyboardInterrupt:
    print('\n⏹️ Stopped')
except Exception as e:
    print(f'\n❌ Error: {e}')
finally:
    print(f'Stats: ok={ok_count} fail={fail_count}')

In [ ]:
# Cell 9 (optional): Test the agent
import requests

# Check health
print('Health:', requests.get(f'{AGENT_URL}/health').json())

# Check world state
world = requests.get(f'{AGENT_URL}/world').json()
print(f'\nWorld state:')
print(f'  Position: {world.get("position")}')
print(f'  Health: {world.get("health")}')
print(f'  Inventory: {len(world.get("inventory", []))} items')
print(f'  Nearby blocks: {len(world.get("nearbyBlocks", []))}')
print(f'  Nearby entities: {world.get("nearbyEntities", [])}')

# Send a goal (uncomment to test)
# print('\nGoal:', requests.post(f'{AGENT_URL}/goal', json={'goal': 'Collect 5 oak logs', 'max_actions': 20}).json())
# time.sleep(30)
# print('Log:', requests.get(f'{AGENT_URL}/log').json())